In [1]:
import pandas as pd
import os

In [6]:
def convert_meta_name(df):
    df = df.rename(columns={'WMO_ID':'wmo_id', 'NAME':'name', 'CURRENT_LATITUDE':'latitude', 'CURRENT_LONGITUDE':'longitude', 
            'PROVINSI':'provinsi', 'KABUPATEN':'regency', 'ELEVATION':'elevation', 'DATA_TIMESTAMP':'time'})
    return df

In [ ]:
DataQc    = '../data/01.QC_Dataset_Level_01'
DataHomo  = '../data/04.Dataset_Final'
params    = ['TEMPERATURE_AVG_C', 'TEMP_24H_TN_C', 'TEMP_24H_TX_C','RAINFALL_24H_MM']  # Parameter yang ingin diproses
baselines = ['1991','1981']                                          # Baseline yang digunakan

def get_qclevel1_dataset(path,param):
    datapath     = os.path.join(path, param, '06.Adjusted')
    df = pd.read_csv(
        os.path.join(datapath, f'{param}_FINAL_QC_DATA_LEVEL1.csv'),
        low_memory=False )
    qc_col = f'QC_{param}'
    Dataset = df[['WMO_ID', 'NAME', 'CURRENT_LATITUDE', 'CURRENT_LONGITUDE', 
                        'PROVINSI', 'KABUPATEN', 'ELEVATION', 'DATA_TIMESTAMP', 
                        qc_col]].copy()
    Dataset = convert_meta_name(Dataset) # Fungsi custom Anda
    Dataset['parameter'] = param
    Dataset['baseline']  = None
    Dataset['source']    = 'qc'
    Dataset = Dataset.rename(columns={qc_col: 'value', 'DATA_TIMESTAMP': 'time'})
    return Dataset

dataQc = pd.DataFrame()
for param in params:
    for baseline in baselines:
        data  = get_qclevel1_dataset(DataQc, param)
        dataQc = pd.concat([dataQc, data], ignore_index=True)

In [ ]:
def get_homgenized_dataset(path,param, baseline):
    df       = pd.read_csv(os.path.join(path, f'{param}_homogen_final_baseline_{baseline}.csv'))
    Dataset  = df[['WMO_ID', 'NAME', 'CURRENT_LATITUDE', 'CURRENT_LONGITUDE', 
                'PROVINSI', 'KABUPATEN', 'ELEVATION', 'DATA_TIMESTAMP', 
                f'HOMO_{param}']].copy()
    Dataset  = convert_meta_name(Dataset)
    Dataset['parameter'] = param
    Dataset['baseline']  = baseline
    Dataset = Dataset.rename(columns={f'HOMO_{param}': 'value'})
    Dataset['source'] = 'homo'
    return Dataset

dataAll = pd.DataFrame()
for param in params:
    for baseline in baselines:
        data  = get_homgenized_dataset(DataHomo, param, baseline)
        dataAll = pd.concat([dataAll, data], ignore_index=True)

,wmo_id,name,latitude,longitude,provinsi,regency,elevation,data_timestamp,value,parameter,baseline,source
0,96001,Stasiun Meteorologi Maimun Saleh,5.87655,95.33785,Nanggroe Aceh Darussalam,Kota Sabang,126.0,1991-01-01,26.650000,TEMPERATURE_AVG_C,1991,homo
1,96001,Stasiun Meteorologi Maimun Saleh,5.87655,95.33785,Nanggroe Aceh Darussalam,Kota Sabang,126.0,1991-01-02,26.950000,TEMPERATURE_AVG_C,1991,homo
2,96001,Stasiun Meteorologi Maimun Saleh,5.87655,95.33785,Nanggroe Aceh Darussalam,Kota Sabang,126.0,1991-01-03,25.350000,TEMPERATURE_AVG_C,1991,homo
3,96001,Stasiun Meteorologi Maimun Saleh,5.87655,95.33785,Nanggroe Aceh Darussalam,Kota Sabang,126.0,1991-01-04,26.700000,TEMPERATURE_AVG_C,1991,homo
4,96001,Stasiun Meteorologi Maimun Saleh,5.87655,95.33785,Nanggroe Aceh Darussalam,Kota Sabang,126.0,1991-01-05,26.750000,TEMPERATURE_AVG_C,1991,homo
...,...,...,...,...,...,...,...,...,...,...,...,...
9286060,97900,Stasiun Meteorologi Mathilda Batlayeri,-7.98000,131.30000,Maluku,Kab. Kep. Tanimbar,24.0,2026-01-28,32.900000,TEMP_24H_TX_C,1981,homo
9286061,97900,Stasiun Meteorologi Mathilda Batlayeri,-7.98000,131.30000,Maluku,Kab. Kep. Tanimbar,24.0,2026-01-29,31.000000,TEMP_24H_TX_C,1981,homo
9286062,97900,Stasiun Meteorologi Mathilda Batlayeri,-7.98000,131.30000,Maluku,Kab. Kep. Tanimbar,24.0,2026-01-30,31.233333,TEMP_24H_TX_C,1981,homo
9286063,97900,Stasiun Meteorologi Mathilda Batlayeri,-7.98000,131.30000,Maluku,Kab. Kep. Tanimbar,24.0,2026-01-31,31.466667,TEMP_24H_TX_C,1981,homo
